# 01 — ZIT only

ZITboost 단독 (joint EM, π·μ·φ 동시 학습) HPO + refit + 후처리.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/final/zit_only/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` (strategy_common.md §1) — 1회 사전 적용, trial 내 미흔듦
- **HPO**: 80 trial, anchor=trial 99 best (1차), narrow ±30% (strategy.md §6 표 그대로), **anchor 첫 trial enqueue**
- **τ_π**: HP로 탐색 (0.84~1.0)
- **후처리**: 집계 8종, position Optuna 50t, zero_clip log space, π threshold SKIP (τ_π가 die-level 역할)

## 1. 환경 설정 + import

Colab/Local 자동 감지. Colab 사용 시 `GDRIVE_MODELING_ID` 채울 것 (modeling.zip 공유 ID — 신규 `3_modeling/`을 zip으로 업로드 후 ID 입력).

In [ ]:
import os, sys

GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'
GDRIVE_MODELING_ID     = ''  # ★ Colab 사용 시 신규 modeling.zip ID 입력

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/zit.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# strategy_common.md §22 — 2_preprocessing 직접 import
PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)

MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo, postprocess
from modules.zit import ZITboostRegressor

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold

import logging, time
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

## 2. 실험 설정

- `N_JOBS = 7` (학원 14코어 / 노트북 2개 병렬, strategy_common §8)
- `PP_FIXED` (strategy_common §1)
- `ZIT_ONLY_ANCHOR` (strategy.md §4 — trial 99 best)
- `ZIT_SEARCH` (strategy.md §6 표 그대로)

In [ ]:
# ── 실험 식별 ──
EXP_ID = 'zit-only-final-001'
USER   = 'jh'

# ── Optuna 예산 ──
N_TRIALS         = 80
TIMEOUT_SEC         = None  # ★ Colab 타임아웃 대비, 초 단위 (None=무제한)
N_FOLDS          = 5
N_JOBS           = 7   # ★ 단일 변수 (strategy_common §8)
N_STARTUP_TRIALS = 8

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, 'final', 'zit_only')
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

# ── y 극단값 clip ──
CLIP_Y_EXTREME = True

# ── PP_FIXED (strategy_common.md §1) ──
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# ── ZIT_ONLY_ANCHOR (strategy.md §4 — trial 99 best, OOF=0.005496) ──
ZIT_ONLY_ANCHOR = {
    'zeta':                  1.149,
    'n_em_iters':            13,
    'mu_n_estimators':       240,
    'mu_learning_rate':      0.00309,
    'mu_num_leaves':         212,
    'mu_max_depth':          3,
    'mu_min_child_samples':  132,
    'mu_subsample':          0.649,
    'mu_colsample_bytree':   0.255,
    'mu_reg_alpha':          0.00576,
    'mu_reg_lambda':         0.00155,
    'pi_n_estimators':       125,
    'pi_learning_rate':      0.0408,
    'pi_num_leaves':         165,
    'pi_max_depth':          11,
    'pi_min_child_samples':  38,
    'phi_n_estimators':      57,
    'phi_learning_rate':     0.00628,
    'phi_num_leaves':        65,
    'phi_max_depth':         4,
    'phi_min_child_samples': 190,
}
ANCHOR_TAU_PI = 0.944

# ── ZIT_SEARCH (strategy.md §6 표 그대로 — narrow ±30%) ──
ZIT_SEARCH = {
    'zeta':                  {'type': 'float', 'low': 1.05,   'high': 1.50,  'log': False},
    'n_em_iters':            {'type': 'int',   'low': 10,     'high': 20},
    'mu_n_estimators':       {'type': 'int',   'low': 170,    'high': 320},
    'mu_learning_rate':      {'type': 'float', 'low': 0.0021, 'high': 0.0046, 'log': True},
    'mu_num_leaves':         {'type': 'int',   'low': 148,    'high': 280},
    'mu_max_depth':          {'type': 'int',   'low': 3,      'high': 5},
    'mu_min_child_samples':  {'type': 'int',   'low': 90,     'high': 180},
    'mu_subsample':          {'type': 'float', 'low': 0.50,   'high': 0.85,  'log': False},
    'mu_colsample_bytree':   {'type': 'float', 'low': 0.18,   'high': 0.35,  'log': False},
    'mu_reg_alpha':          {'type': 'float', 'low': 0.002,  'high': 0.015, 'log': True},
    'mu_reg_lambda':         {'type': 'float', 'low': 5e-4,   'high': 5e-3,  'log': True},
    'pi_n_estimators':       {'type': 'int',   'low': 90,     'high': 175},
    'pi_learning_rate':      {'type': 'float', 'low': 0.028,  'high': 0.060, 'log': True},
    'pi_num_leaves':         {'type': 'int',   'low': 115,    'high': 220},
    'pi_max_depth':          {'type': 'int',   'low': 8,      'high': 13},
    'pi_min_child_samples':  {'type': 'int',   'low': 25,     'high': 55},
    'phi_n_estimators':      {'type': 'int',   'low': 40,     'high': 80},
    'phi_learning_rate':     {'type': 'float', 'low': 0.004,  'high': 0.010, 'log': True},
    'phi_num_leaves':        {'type': 'int',   'low': 45,     'high': 90},
    'phi_max_depth':         {'type': 'int',   'low': 3,      'high': 6},
    'phi_min_child_samples': {'type': 'int',   'low': 130,    'high': 260},
}
TAU_PI_RANGE = (0.84, 1.0)   # anchor 0.944 ± 0.10 / 1.0 (strategy.md §8)

print(f'EXP_ID={EXP_ID} | USER={USER}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DB_PATH={DB_PATH}')
print(f'ZIT search HP={len(ZIT_SEARCH)} + tau_pi={1} = {len(ZIT_SEARCH)+1}')

## 3. 데이터 로드 + PP_FIXED 사전 적용 (1회)

PP_FIXED는 trial 내에서 안 흔드므로 study 시작 전 1회만 적용하고 그 결과를 모든 trial에서 공유.

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

# PP_FIXED 1회 적용
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# numpy 변환 (학습 가속)
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)

uid_train_die = xs_train[KEY_COL].values
uid_val_die   = xs_val[KEY_COL].values
uid_test_die  = xs_test[KEY_COL].values

y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)

print(f'[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'  unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')

## 4. K-fold split + Optuna objective

- KFold는 **unit ID 단위 분할** (strategy_common §6) — 같은 unit의 4 die는 같은 fold
- 매 trial: 5 fold 학습 + die-level π/μ → τ_π 적용 → unit 평균 → OOF unit RMSE
- pruning: MedianPruner(n_warmup_steps=2)

In [ ]:
unique_units = y_train_unit_s.index.values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(unique_units))

def _mean_die_to_unit(pred_die, uid_die):
    df = pd.DataFrame({KEY_COL: uid_die, 'pred': pred_die})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()

def _apply_tau_pi(pred_die, pi_die, tau_pi):
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def objective(trial):
    t0 = time.time()
    params = hpo.sample_from_space(trial, ZIT_SEARCH)
    tau_pi = trial.suggest_float('tau_pi', TAU_PI_RANGE[0], TAU_PI_RANGE[1])

    # ZIT 모델 인자 보강
    params['random_state'] = SEED
    params['n_jobs']       = N_JOBS
    params['verbose']      = -1
    params['device']       = 'cpu'
    params['em_tol']       = 1e-7

    fold_oof_rmse = []
    oof_pred_unit = pd.Series(np.nan, index=y_train_unit_s.index, dtype=np.float64)

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        model = ZITboostRegressor(**params)
        model.fit(X_train[tr_mask], y_train_die[tr_mask])

        pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
        pred_die_raw = np.clip((1 - pi_vl) * mu_vl, 0, None)
        pred_die_taupi = _apply_tau_pi(pred_die_raw, pi_vl, tau_pi)
        unit_pred_df = _mean_die_to_unit(pred_die_taupi, uid_train_die[vl_mask])

        oof_pred_unit.loc[unit_pred_df[KEY_COL].values] = unit_pred_df['pred'].values
        y_vl = y_train_unit_s.loc[unit_pred_df[KEY_COL].values].values
        fold_rmse = float(np.sqrt(np.mean((unit_pred_df['pred'].values - y_vl) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        avg = float(np.mean(fold_oof_rmse))
        trial.report(avg, step=fold_idx)
        if trial.should_prune():
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', time.time() - t0)
            trial.set_user_attr('tau_pi', tau_pi)
            raise optuna.TrialPruned()

    if oof_pred_unit.isna().any():
        raise RuntimeError('OOF NaN — fold 누락')

    oof_rmse = float(np.sqrt(np.mean((oof_pred_unit.values - y_train_unit_s.values) ** 2)))
    elapsed = time.time() - t0
    trial.set_user_attr('elapsed_sec', elapsed)
    trial.set_user_attr('tau_pi', tau_pi)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    print(f'  trial #{trial.number}: τ_π={tau_pi:.3f}, oof={oof_rmse:.6f}, elapsed={elapsed:.0f}s')
    return oof_rmse


print(f'fold split: {N_FOLDS} folds, unit 단위 분할, seed={SEED}')

## 5. Optuna study 생성 + anchor enqueue + optimize

- TPESampler(seed=None, multivariate=True, group=True) — strategy_common §4
- `enqueue_anchor`로 첫 trial은 1차 best HP 그대로 (strategy_common §5)

In [ ]:
sampler = TPESampler(
    seed=None,
    multivariate=True,
    group=True,
    n_startup_trials=N_STARTUP_TRIALS,
)
pruner = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=2)

study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler,
    pruner=pruner,
    direction='minimize',
    load_if_exists=True,
)

# anchor 첫 trial 강제 (strategy_common §5)
ANCHOR_FOR_ENQUEUE = dict(ZIT_ONLY_ANCHOR)
ANCHOR_FOR_ENQUEUE['tau_pi'] = ANCHOR_TAU_PI
if len(study.trials) == 0:
    hpo.enqueue_anchor(study, ANCHOR_FOR_ENQUEUE)
else:
    print(f'[enqueue skip] 기존 trial {len(study.trials)} 있음 — resume')

study_meta = {
    'exp_id': EXP_ID, 'user': USER, 'model': 'ZITboost (zit_only)',
    'n_trials': N_TRIALS, 'n_folds': N_FOLDS, 'n_jobs': N_JOBS,
    'pp_fixed': PP_FIXED, 'anchor': ZIT_ONLY_ANCHOR, 'anchor_tau_pi': ANCHOR_TAU_PI,
    'sampler': 'TPE seed=None multivariate group',
    'pruner':  f'MedianPruner n_startup={N_STARTUP_TRIALS} n_warmup=2',
    'CLIP_Y_EXTREME': CLIP_Y_EXTREME, 'SEED': int(SEED),
}
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}, DB: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')

t_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=1, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_start:.0f}s, total trials={len(study.trials)}')
print(f'  best OOF RMSE: {study.best_value:.6f}')

## 6. Best trial 정보 + anchor enqueue 검증

trial 0이 anchor와 일치하는지 확인 (strategy.md §12 검증 항목).

In [ ]:
best_trial = study.best_trial
best_params = best_trial.params
best_tau_pi = float(best_trial.user_attrs.get('tau_pi', ANCHOR_TAU_PI))

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE  : {best_trial.value:.6f}')
print(f'  best τ_π  : {best_tau_pi:.4f}')
print(f'  elapsed   : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')
for k, v in sorted(best_params.items()):
    print(f'    {k}: {v}')

# enqueue 검증 — trial 0이 anchor와 일치?
trial0 = study.trials[0]
anchor_check = all(
    abs(float(trial0.params.get(k, np.nan)) - float(v)) < 1e-9
    if not isinstance(v, str) else trial0.params.get(k) == v
    for k, v in ANCHOR_FOR_ENQUEUE.items()
)
print(f'\n[검증] trial 0 == anchor? {anchor_check}')

## 7. Best HP 5-fold refit + die-level π/μ 캡처

In [ ]:
best_full_params = {k: v for k, v in best_params.items() if k != 'tau_pi'}
best_full_params['random_state'] = SEED
best_full_params['n_jobs']       = N_JOBS
best_full_params['verbose']      = -1
best_full_params['device']       = 'cpu'
best_full_params['em_tol']       = 1e-7

n_train_die = len(X_train)
n_val_die   = len(X_val)
n_test_die  = len(X_test)

oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)

val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)
test_die_pi   = np.zeros(n_test_die)
test_die_mu   = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

fold_models = []
em_history_per_fold = []

print(f'=== Best HP 5-fold refit ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unique_units[tr_uidx]
    vl_units = unique_units[vl_uidx]
    tr_mask = np.isin(uid_train_die, tr_units)
    vl_mask = np.isin(uid_train_die, vl_units)

    model = ZITboostRegressor(**best_full_params)
    model.fit(X_train[tr_mask], y_train_die[tr_mask])

    pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
    oof_die_pi[vl_mask]   = pi_vl
    oof_die_mu[vl_mask]   = mu_vl
    oof_die_pred[vl_mask] = np.clip((1 - pi_vl) * mu_vl, 0, None)

    pi_v, mu_v, _ = model.predict_components(X_val)
    pi_t, mu_t, _ = model.predict_components(X_test)
    val_die_pi    += pi_v / N_FOLDS
    val_die_mu    += mu_v / N_FOLDS
    val_die_pred  += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi   += pi_t / N_FOLDS
    test_die_mu   += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    fold_models.append(model)
    em_history_per_fold.append(model.em_history_)
    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

# EM 수렴 체크 (strategy.md §12)
# ZITboost(부모) em_history는 'rmse' 키 (die-level), BagZIT는 'unit_rmse' 추가 — 자동 감지
print(f'\n[EM 수렴 체크]')
for f, hist in enumerate(em_history_per_fold):
    key = 'unit_rmse' if hist and 'unit_rmse' in hist[0] else 'rmse'
    rmses = [h[key] for h in hist]
    monotonic = all(rmses[i+1] <= rmses[i] + 1e-6 for i in range(len(rmses)-1))
    print(f'  fold {f+1}: {len(hist)} EM iter, last_{key}={rmses[-1]:.6f}, monotonic_decreasing={monotonic}')

print(f'\n[refit 완료] die-level π/μ/pred 캡처 OK')

## 8. 후처리 — τ_π 적용 → 집계 8 + position Optuna + zero_clip(log)

- 분류 threshold (§9): SKIP (τ_π가 die-level 역할)
- 집계 다양성 (§10): 8종 (Q25/Q75 추가)
- Position 가중치 (§11): Optuna sub-study 50 trial
- zero_clip (§12): log space 비교

In [ ]:
# τ_π die-level 적용
oof_die_pred_taupi  = _apply_tau_pi(oof_die_pred,  oof_die_pi,  best_tau_pi)
val_die_pred_taupi  = _apply_tau_pi(val_die_pred,  val_die_pi,  best_tau_pi)
test_die_pred_taupi = _apply_tau_pi(test_die_pred, test_die_pi, best_tau_pi)

killed = {
    'oof':  float((oof_die_pi  > best_tau_pi).mean()),
    'val':  float((val_die_pi  > best_tau_pi).mean()),
    'test': float((test_die_pi > best_tau_pi).mean()),
}
print(f'[τ_π={best_tau_pi:.3f} 적용] 0 처리 die 비율: {killed}')

# 후처리 호출 — strategy_common.md §10·§11·§12: train OOF best 후보 → val 적용 → val 개선 시만 채택
pp_res = postprocess.tune_and_apply(
    xs_train, xs_val, xs_test,
    die_pred_train=oof_die_pred_taupi,
    die_pred_val=val_die_pred_taupi,
    die_pred_test=test_die_pred_taupi,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],     # ★ val 비교용
    use_pi_threshold=False,                # ZIT τ_π로 die-level 처리됨
    agg_methods=postprocess.AGG_METHODS,   # 8종
    zero_clip_log_space=True,              # strategy_common §12
    position_method='optuna',
    position_optuna_n_trials=50,
)

print(f'\n[Postprocess]')
print(f'  best_agg            : {pp_res["best_agg"]}')
print(f'  pos_weights         : {pp_res["pos_weights"]}')
print(f'  best_zero_clip(log) : {pp_res["best_zero_clip"]}')
print(f'  position_method     : {pp_res["position_method"]}')
print(f'  train_rmse          : {pp_res["train_rmse"]:.6f}')
print(f'  val_rmse_final      : {pp_res.get("val_rmse_final")}')
print(f'  agg_rmses           : {pp_res["agg_rmses"]}')
print(f'  decisions           :')
for k, v in pp_res.get('decisions', {}).items():
    print(f'    {k:14s} {v}')

## 9. 산출물 9개 저장 (strategy_common §15)

best_params.json + fold_models.pkl + 6 CSV (die ×3 + unit ×3) + optuna_*.db

In [ ]:
import json, pickle, hashlib

# 1) fold_models.pkl
with open(os.path.join(OUT_DIR, 'fold_models.pkl'), 'wb') as f:
    pickle.dump({
        'fold_models':         fold_models,
        'feature_names':       feat_cols_clean,
        'model_name':          'zitboost',
        'n_folds':             N_FOLDS,
        'em_history_per_fold': em_history_per_fold,
    }, f)

# 2) best_params.json
uid_arr = ys_input['train'][KEY_COL].unique()
unit_ids_hash = hashlib.sha1(','.join(map(str, uid_arr)).encode()).hexdigest()

best_meta = {
    'exp_id':                EXP_ID,
    'model_name':            'zitboost',
    'best_trial_number':     best_trial.number,
    'best_oof_rmse':         float(best_trial.value),
    'best_params_resolved':  best_full_params,
    'best_tau_pi':           best_tau_pi,
    'feature_names':         feat_cols_clean,
    'n_features':            len(feat_cols_clean),
    'n_folds':               N_FOLDS,
    'unit_ids_hash':         unit_ids_hash,
    'n_units_train':         int(len(uid_arr)),
    'effective_pp_params':   PP_FIXED,
    'study_meta':            study_meta,
    'postprocess': {
        'best_agg':            pp_res['best_agg'],
        'pos_weights':         pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
        'best_zero_clip':      float(pp_res['best_zero_clip']),
        'zero_clip_log_space': pp_res['zero_clip_log_space'],
        'position_method':     pp_res['position_method'],
        'agg_rmses':           {k: float(v) for k, v in pp_res['agg_rmses'].items()},
        'train_rmse':          float(pp_res['train_rmse']),
    },
}
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump(best_meta, f, indent=2, ensure_ascii=False, default=str)

# 3-5) die-level CSV
def _build_die_df(uid, die_id, position, pi, mu, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL: uid, DIE_KEY_COL: die_id, 'position': position,
        'pi': pi, 'one_minus_pi': 1.0 - pi, 'mu': mu, 'pred': pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

_build_die_df(
    uid_train_die, xs_train[DIE_KEY_COL].values, xs_train['position'].values,
    oof_die_pi, oof_die_mu, oof_die_pred, y_train_unit_s,
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(
    uid_val_die, xs_val[DIE_KEY_COL].values, xs_val['position'].values,
    val_die_pi, val_die_mu, val_die_pred, y_val_unit_s,
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(
    uid_test_die, xs_test[DIE_KEY_COL].values, xs_test['position'].values,
    test_die_pi, test_die_mu, test_die_pred, y_test_unit_s,
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

# 6-8) unit-level CSV (postprocess 적용본 + health merge)
def _build_unit_df(unit_pred_df, y_unit):
    out = unit_pred_df.copy()
    out[TARGET_COL] = out[KEY_COL].map(y_unit)
    return out

_build_unit_df(pp_res['final_train_unit'], y_train_unit_s).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(pp_res['final_val_unit'],   y_val_unit_s  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(pp_res['final_test_unit'],  y_test_unit_s ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# 9) optuna_*.db는 study.optimize에서 자동 저장

# 산출물 확인
print(f'\n저장 완료: {OUT_DIR}')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1024
    print(f'  {fn:30s}  {sz:10,.1f} KB')

# Colab → 로컬 zip 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'zit_only_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip))
except ImportError:
    pass